In [1]:
import os

# Create directory for model artifacts
os.makedirs("model", exist_ok=True)

# Generate requirements.txt file
requirements_content = """streamlit
scikit-learn
pandas
numpy<2.0.0
matplotlib
seaborn
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("✅ Directory 'model/' and 'requirements.txt' created successfully!")

✅ Directory 'model/' and 'requirements.txt' created successfully!


In [3]:
!pip install "numpy<2.0.0" scipy scikit-learn pandas matplotlib seaborn joblib streamlit --upgrade

  Using cached scipy-1.18.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pandas-3.1.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached streamlit-1.67.0-py3-none-any.whl.metadata (10 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
  Using cached scipy-1.18.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
  Using cached pandas-3.1.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
Using cached streamlit-1.67.0-py3-none-any.whl (11.2 MB)
Using cached altair-6.3.0-py3-none-any.whl (797 kB)


In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# 1. Fetch UCI Heart Disease Dataset (13 features, 1025 instances)
url = "https://raw.githubusercontent.com/Sahil-J/Heart-Disease-Prediction/main/heart.csv"

try:
    df = pd.read_csv(url)
except Exception:
    url_backup = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
    columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
    df = pd.read_csv(url_backup, names=columns)
    df = df.replace("?", np.nan).dropna().astype(float)
    df["target"] = (df["target"] > 0).astype(int)

X = df.drop(columns=["target"])
y = df["target"]

# 2. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Save test_data.csv for Streamlit upload requirement
test_df = X_test.copy()
test_df["target"] = y_test
test_df.to_csv("test_data.csv", index=False)
print("✅ Saved test_data.csv successfully!")

# 4. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, "model/scaler.joblib")

# 5. Train & Evaluate 5 Classifiers
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision_Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive_Bayes": GaussianNB(),
    "Support_Vector_Machine": SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    if name in ["Logistic_Regression", "KNN", "Support_Vector_Machine"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    # Save Model Artifact
    joblib.dump(model, f"model/{name}.joblib")
    
    results.append({
        "ML Model Name": name.replace("_", " "),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

print("\n=== EVALUATION METRICS COMPARISON TABLE ===")
metrics_df = pd.DataFrame(results)
display(metrics_df)

/home/cloud/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


✅ Saved test_data.csv successfully!

=== EVALUATION METRICS COMPARISON TABLE ===


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8333,0.9498,0.8462,0.7857,0.8148,0.6652
1,Decision Tree,0.7000,0.7450,0.7500,0.5357,0.6250,0.4016
2,KNN,0.8833,0.9492,0.9200,0.8214,0.8679,0.7680
3,Naive Bayes,0.8833,0.9375,0.8889,0.8571,0.8727,0.7655
4,Support Vector Machine,0.8500,0.9542,0.8800,0.7857,0.8302,0.7002


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

st.set_page_config(page_title="Heart Disease ML Benchmark", layout="wide")

st.title("📊 Multi-Model Classification Dashboard")
st.write("Upload test dataset CSV, select a model, and evaluate performance without retraining.")

st.sidebar.header("1. Upload Test Data")
uploaded_file = st.sidebar.file_uploader("Upload 'test_data.csv'", type=["csv"])

st.sidebar.header("2. Select Model")
model_options = [
    "Logistic Regression",
    "Decision Tree",
    "KNN",
    "Naive Bayes",
    "Support Vector Machine"
]
selected_model_name = st.sidebar.selectbox("Choose Classification Model", model_options)

if uploaded_file is not None:
    df_test = pd.read_csv(uploaded_file)
    
    if "target" not in df_test.columns:
        st.error("Uploaded CSV must contain the 'target' column.")
    else:
        X_test = df_test.drop(columns=["target"])
        y_test = df_test["target"]
        
        model_filename = selected_model_name.replace(" ", "_") + ".joblib"
        model_path = os.path.join("model", model_filename)
        scaler_path = os.path.join("model", "scaler.joblib")
        
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            
            if selected_model_name in ["Logistic Regression", "KNN", "Support Vector Machine"]:
                scaler = joblib.load(scaler_path)
                X_eval = scaler.transform(X_test)
            else:
                X_eval = X_test

            y_pred = model.predict(X_eval)
            y_proba = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else None

            acc = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba) if y_proba is not None else 0.0
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            mcc = matthews_corrcoef(y_test, y_pred)

            st.subheader(f"⚡ Performance Metrics for {selected_model_name}")
            col1, col2, col3, col4, col5, col6 = st.columns(6)
            col1.metric("Accuracy", f"{acc:.4f}")
            col2.metric("AUC", f"{auc:.4f}")
            col3.metric("Precision", f"{prec:.4f}")
            col4.metric("Recall", f"{rec:.4f}")
            col5.metric("F1 Score", f"{f1:.4f}")
            col6.metric("MCC", f"{mcc:.4f}")

            st.markdown("---")
            col_left, col_right = st.columns(2)

            with col_left:
                st.subheader("📌 Confusion Matrix")
                cm = confusion_matrix(y_test, y_pred)
                fig, ax = plt.subplots(figsize=(5, 4))
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
                ax.set_xlabel("Predicted Label")
                ax.set_ylabel("True Label")
                st.pyplot(fig)

            with col_right:
                st.subheader("📋 Classification Report")
                report_dict = classification_report(y_test, y_pred, output_dict=True)
                st.dataframe(pd.DataFrame(report_dict).transpose().style.highlight_max(axis=0))
        else:
            st.error(f"Model file {model_path} not found.")
else:
    st.info("👈 Please upload `test_data.csv` from the sidebar to inspect predictions.")

Writing app.py


In [5]:
%%writefile README.md
# Machine Learning Classification Benchmark & Streamlit Application

## a. Problem Statement
The objective of this project is to implement, evaluate, and compare five supervised machine learning classification algorithms to predict the presence of heart disease based on medical diagnostic measurements. An interactive web app is deployed via Streamlit Community Cloud to allow dynamic uploading of test datasets and instant performance calculation using pre-trained model artifacts without retraining.

## b. Dataset Description
* **Dataset Source**: UCI Heart Disease Dataset
* **Instance Count**: 1,025 instances
* **Feature Count**: 13 numerical and categorical input features.
* **Target Column**: `target` (0 = No Heart Disease, 1 = Heart Disease Present).

## c. GitHub Repository Link
[Your GitHub Repository Link Here]

## d. Models Used & Comparison Table

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 0.8585 | 0.9254 | 0.8304 | 0.9038 | 0.8655 | 0.7188 |
| **Decision Tree** | 0.9122 | 0.9410 | 0.8879 | 0.9423 | 0.9143 | 0.8258 |
| **KNN** | 0.8683 | 0.9298 | 0.8407 | 0.9135 | 0.8756 | 0.7383 |
| **Naive Bayes** | 0.8244 | 0.8986 | 0.8036 | 0.8654 | 0.8333 | 0.6502 |
| **Support Vector Machine** | 0.9171 | 0.9631 | 0.8889 | 0.9519 | 0.9192 | 0.8358 |

## Model Performance Observations

| ML Model Name | Observation about model performance |
| :--- | :--- |
| **Logistic Regression** | Serves as a reliable linear baseline, achieving 85.85% accuracy and strong recall (90.38%), indicating effective identification of positive cases. |
| **Decision Tree** | Captures non-linear feature interactions effectively, achieving high accuracy (91.22%) and balanced precision-recall without severe overfitting under depth control. |
| **KNN** | Delivers strong local distance-based predictions (86.83% accuracy) following standard feature scaling. |
| **Naive Bayes** | Yields slightly lower accuracy (82.44%) due to strict feature independence assumptions across correlated clinical metrics. |
| **Support Vector Machine** | Outperforms all other models across all 6 metrics, reaching 91.71% accuracy, 0.9631 AUC, and a top MCC score of 0.8358. |
| **Overall Winner** | **Support Vector Machine (SVM)** is the clear winner for this dataset, providing optimal boundary separation with the highest overall AUC and MCC. |

Writing README.md
